<a href="https://colab.research.google.com/github/Okahak71/learning_ai/blob/main/makemore/MLP/makemore_MLP_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Trying to beat Andrej's loss function from MLP video

import torch as t
import torch.nn.functional as F
import time

t.set_default_device('cuda')

print_every = 5000
start_time = time.time()
last_time = time.time()

words = open("names.txt", 'r').read().splitlines()
st = sorted(list(set(''.join(words))))
stoi = {s:i + 1 for i, s in enumerate(st)}
stoi["."] = 0
itos = {i : s for s, i in stoi.items()}

#Hyperparameters
learning_rate = 0.1
vocab_size = 27
block_size = 5
neurons = 200
dim = 10

#-------------
def build_dataset(words):
    xs, ys = [], []
    for w in words:
        context = [0] * block_size
        for c in w + '.':
            xs.append(context)
            ys.append(stoi[c])
            context = context[1:] + [stoi[c]]
    xs = t.tensor(xs)
    ys = t.tensor(ys)
    return xs, ys

#Training - 80%, Val/Dev - 10%, Testing - 10%
import random
random.seed(42)
random.shuffle(words)
sz1 = int(0.8 * len(words))
sz2 = int(0.9 * len(words))
x_tr, y_tr = build_dataset(words[:sz1])
x_dev, y_dev = build_dataset(words[sz1:sz2])
x_test, y_test = build_dataset(words[sz2:])

#-----------Parameters---------
rng = t.Generator(device='cuda').manual_seed(2147483647)
C = t.randn((vocab_size, dim), generator=rng) # THE Lookup Table

#first layer
w1 = t.randn((block_size * dim, neurons), generator=rng)
b1 = t.randn((neurons), generator=rng)

#second layer
w2 = t.randn((neurons, vocab_size), generator=rng)
b2 = t.randn(vocab_size, generator=rng)

parameters = [C, w1, b1, w2, b2]

for p in parameters:
    p.requires_grad = True
total = 0

In [ ]:

for i in range(300000):
    total += 1
    # Minibatch construction
    id = t.randint(0, x_tr.shape[0], (256,))

    #forward pass
    emb = C[x_tr[id]]
    h = t.tanh(emb.view(-1, block_size * dim) @ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, y_tr[id])
    # print(loss.item())
    if i < 2:
        print(f"Initial loss = {loss.item()}")

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    lr = 0.1 if i < 120000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    #------------------------------
    if i > 0 and i % print_every == 0:
        current_time = time.time()
        elapsed_chunk = current_time - last_time
        iterations_per_second = print_every / elapsed_chunk
        print(f"\rIteration: {i} | Speed: {iterations_per_second:.2f} it/s | loss = {loss}")
        last_time = current_time

Initial loss = 28.99401092529297
Initial loss = 25.97559356689453
Iteration: 5000 | Speed: 549.53 it/s | loss = 2.345468521118164
Iteration: 10000 | Speed: 736.44 it/s | loss = 2.369703531265259
Iteration: 15000 | Speed: 774.89 it/s | loss = 2.2931132316589355
Iteration: 20000 | Speed: 736.97 it/s | loss = 2.2479963302612305
Iteration: 25000 | Speed: 785.66 it/s | loss = 2.2951159477233887
Iteration: 30000 | Speed: 738.71 it/s | loss = 2.308537483215332
Iteration: 35000 | Speed: 767.52 it/s | loss = 2.423676013946533
Iteration: 40000 | Speed: 670.91 it/s | loss = 2.157349109649658
Iteration: 45000 | Speed: 769.00 it/s | loss = 2.2632055282592773
Iteration: 50000 | Speed: 746.58 it/s | loss = 2.096034049987793
Iteration: 55000 | Speed: 750.74 it/s | loss = 2.024639129638672
Iteration: 60000 | Speed: 743.96 it/s | loss = 2.2385432720184326
Iteration: 65000 | Speed: 725.82 it/s | loss = 2.2156455516815186
Iteration: 70000 | Speed: 778.26 it/s | loss = 2.139007568359375
Iteration: 75000 | 

In [ ]:
emb = C[x_dev]
h = t.tanh(emb.view(-1, dim * block_size) @ w1 + b1)
logits = h @ w2 + b2
loss = F.cross_entropy(logits, y_dev)
print(loss.item())
print(f"current learning_rate : {lr}")
print(total)

2.1386289596557617
current learning_rate : 0.01
300000


In [ ]:
for i in range(50000):
    total += 1
    id = t.randint(0, x_tr.shape[0], (128,))
    emb = C[x_tr[id]]
    h = t.tanh(emb.view(-1, block_size * dim) @ w1 + b1)
    logits = h @ w2 + b2
    loss = F.cross_entropy(logits, y_tr[id])
    for p in parameters:
        p.grad = None
    loss.backward()
    lr = learning_rate / pow(10, (total / 70000))
    for p in parameters:
        p.data += -lr * p.grad
    if i % 5000 == 0 and i:
        print(loss.item())

2.2048277854919434
2.153212547302246
2.3063201904296875
2.131532907485962
2.2017157077789307
2.378223419189453
2.2086703777313232
2.185281276702881
2.3237862586975098
